### The goal of this file is to reproduce Yevgeniy results of L2 vowels as sanity check
Dataset: ALLSTAR (Segmented)

In [ ]:
import os
import re
import pandas as pd
from pathlib import Path

DATA_ROOT = Path("../Data/ALLSSTAR")

records = []
# CCC field can contain alphanumeric codes like HT1, HT2, DHR, NWS, LPP
pattern = re.compile(
    r"^ALL_(\d+)_([MF])_([A-Z]{3})_([A-Z]{3})_([A-Z0-9]{2,3})$"
)

for folder in sorted(DATA_ROOT.iterdir()):
    if not folder.is_dir():
        continue

    wav_lookup = {f.stem: f.resolve() for f in folder.glob("*.wav")}
    tg_lookup = {f.stem: f.resolve() for f in folder.glob("*.TextGrid")}
    all_stems = set(wav_lookup) | set(tg_lookup)

    for stem in sorted(all_stems):
        m = pattern.match(stem)
        if not m:
            print(f"WARNING: could not parse filename: {stem}")
            continue

        participant_id, gender, native_lang, task_lang, task = m.groups()

        records.append({
            "filename_stem": stem,
            "folder": folder.name,
            "participant_id": int(participant_id),
            "gender": gender,
            "native_language": native_lang,
            "task_language": task_lang,
            "task": task,
            "wav_path": str(wav_lookup[stem]) if stem in wav_lookup else None,
            "textgrid_path": str(tg_lookup[stem]) if stem in tg_lookup else None,
        })

file_metadata = pd.DataFrame(records)
file_metadata["has_wav"] = file_metadata["wav_path"].notna()
file_metadata["has_textgrid"] = file_metadata["textgrid_path"].notna()

folder_summary = file_metadata.groupby("folder").agg(
    n_wav=("has_wav", "sum"),
    n_textgrid=("has_textgrid", "sum"),
    n_total=("filename_stem", "count"),
).astype(int)

print(f"Total file entries: {len(file_metadata)}")
print(f"Unique participants: {file_metadata['participant_id'].nunique()}")
print(f"Native languages: {sorted(file_metadata['native_language'].unique())}")
print(f"Task languages: {sorted(file_metadata['task_language'].unique())}")
print(f"Tasks: {sorted(file_metadata['task'].unique())}")
print(f"\nTextGrid coverage: {file_metadata['has_textgrid'].sum()} / {len(file_metadata)}")
print(f"WAV coverage: {file_metadata['has_wav'].sum()} / {len(file_metadata)}")
print(f"\nGender distribution:\n{file_metadata.drop_duplicates('participant_id')['gender'].value_counts().to_string()}")
print(f"\nFiles per native language:\n{file_metadata['native_language'].value_counts().sort_index().to_string()}")
print(f"\nPer-folder file counts ({len(folder_summary)} folders):")
folder_summary
file_metadata.head(30)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Build unique folder-level combos: L1 / Task Language / Task
# Each folder = one unique (native_language, task_language, task) triple
folder_combos = (
    file_metadata[["native_language", "task_language", "task"]]
    .drop_duplicates()
    .sort_values(["native_language", "task_language", "task"])
)

w_l1 = widgets.Dropdown(
    options=sorted(file_metadata["native_language"].unique()),
    description="L1 (native):",
)
w_tasklang = widgets.Dropdown(options=[], description="Task lang:")
w_task = widgets.Dropdown(options=[], description="Task:")
w_fileid = widgets.Dropdown(options=[], description="File ID:")

def update_tasklangs(*_):
    mask = folder_combos["native_language"] == w_l1.value
    opts = sorted(folder_combos.loc[mask, "task_language"].unique())
    w_tasklang.options = opts

def update_tasks(*_):
    mask = (
        (folder_combos["native_language"] == w_l1.value) &
        (folder_combos["task_language"] == w_tasklang.value)
    )
    opts = sorted(folder_combos.loc[mask, "task"].unique())
    w_task.options = opts

def update_fileids(*_):
    mask = (
        (file_metadata["native_language"] == w_l1.value) &
        (file_metadata["task_language"] == w_tasklang.value) &
        (file_metadata["task"] == w_task.value)
    )
    stems = sorted(file_metadata.loc[mask, "filename_stem"].unique())
    w_fileid.options = stems

w_l1.observe(update_tasklangs, names="value")
w_tasklang.observe(update_tasks, names="value")
w_task.observe(update_fileids, names="value")

# Initialize cascade
update_tasklangs()
update_tasks()
update_fileids()

display(
    widgets.HBox([w_l1, w_tasklang, w_task]),
    w_fileid,
)

In [ ]:
# ── Step 1: Parse the selected TextGrid and show all tier/interval info ──

import re as _re

def parse_textgrid(path):
    """Parse a Praat long-format TextGrid into a list of tier dicts.

    Each tier dict has keys:
        class, name, xmin, xmax,
        intervals: list of {xmin, xmax, text}
    """
    for enc in ("utf-8", "utf-16", "utf-16-le", "latin-1"):
        try:
            with open(path, "r", encoding=enc) as f:
                raw = f.read()
            break
        except (UnicodeDecodeError, UnicodeError):
            continue

    tiers = []
    # Split on tier headers
    tier_blocks = _re.split(r'item\s*\[\d+\]\s*:', raw)
    for block in tier_blocks[1:]:  # skip preamble before first item
        tier = {}
        tier["class"] = _re.search(r'class\s*=\s*"([^"]+)"', block).group(1)
        tier["name"] = _re.search(r'name\s*=\s*"([^"]*)"', block).group(1)
        tier["xmin"] = float(_re.search(r'(?<!\[)xmin\s*=\s*(-?[\d.]+)', block).group(1))
        tier["xmax"] = float(_re.search(r'(?<!\[)xmax\s*=\s*(-?[\d.]+)', block).group(1))

        intervals = []
        for m in _re.finditer(
            r'intervals\s*\[\d+\]\s*:\s*'
            r'xmin\s*=\s*(-?[\d.]+)\s*'
            r'xmax\s*=\s*(-?[\d.]+)\s*'
            r'text\s*=\s*"([^"]*)"',
            block,
        ):
            intervals.append({
                "xmin": float(m.group(1)),
                "xmax": float(m.group(2)),
                "text": m.group(3).strip(),
            })
        tier["intervals"] = intervals
        tiers.append(tier)
    return tiers

# ── Look up the selected file and parse ──
selected_stem = w_fileid.value
row = file_metadata[file_metadata["filename_stem"] == selected_stem].iloc[0]
tg_path = row["textgrid_path"]

print(f"Selected file : {selected_stem}")
print(f"TextGrid path : {tg_path}")
print()

tiers = parse_textgrid(tg_path)
print(f"Number of tiers: {len(tiers)}")
print("=" * 70)

for i, tier in enumerate(tiers):
    print(f"\n── Tier {i+1}: \"{tier['name']}\" ({tier['class']}) ──")
    print(f"   Time range : {tier['xmin']:.3f} – {tier['xmax']:.3f} s")
    print(f"   # intervals: {len(tier['intervals'])}")
    # Show first 15 non-empty intervals
    nonempty = [iv for iv in tier['intervals'] if iv['text']]
    print(f"   # non-empty : {len(nonempty)}")
    print(f"   First 15 non-empty intervals:")
    for iv in nonempty[:15]:
        dur_ms = (iv['xmax'] - iv['xmin']) * 1000
        print(f"      {iv['xmin']:8.3f} – {iv['xmax']:8.3f}  ({dur_ms:6.1f} ms)  \"{iv['text']}\"")
    if len(nonempty) > 15:
        print(f"      ... and {len(nonempty) - 15} more")

# ── Identify the phone tier (the one we need for formant analysis) ──
phone_tier = None
for tier in tiers:
    if "phone" in tier["name"].lower():
        phone_tier = tier
        break
if phone_tier is None:
    phone_tier = tiers[-1]  # fallback: last tier is usually phones

phone_intervals = [iv for iv in phone_tier["intervals"] if iv["text"] not in ("", "sil", "sp", "SIL", "SP")]
print("\n" + "=" * 70)
print(f"Phone tier selected: \"{phone_tier['name']}\"")
print(f"Total phoneme tokens (excl. silence/pause): {len(phone_intervals)}")
unique_phones = sorted(set(iv["text"] for iv in phone_intervals))
print(f"Unique phone labels ({len(unique_phones)}): {unique_phones}")

In [ ]:
# ── Step 2: Extract each phoneme from the WAV and label vowel vs consonant ──
#
# Unified vowel inventory across ALL 18 L1-L2 pairs in ALLSSTAR.
# Derived by scanning one sample TextGrid per pair (see findings below).
#
# ┌──────────────────────────────────────────────────────────────────────────────┐
# │ Tier structure conventions discovered:                                      │
# │                                                                             │
# │ ALL *_ENG_* files (3 tiers):                                                │
# │   "utt" / "Speaker - word" / "Speaker - phone"   ← ARPAbet labels          │
# │                                                                             │
# │ ALL *_L1_* files where L1≠ENG (2 or 3 tiers):                              │
# │   "sentence" or "utt" / ["sentence - words"] / "sentence - phones"          │
# │   Phone labels vary per language — see tables below.                        │
# │                                                                             │
# │ CMN_CMN is the only 2-tier case: "sentence" + "sentence - phones"           │
# │ All others (L1_L1 and L1_ENG) have 3 tiers.                                │
# └──────────────────────────────────────────────────────────────────────────────┘
#
# ┌─────────────────────────────────────────────────────────────────────────────────┐
# │ ENG task (ARPAbet): all L1_ENG pairs share the same ~54-55 phone set           │
# │   Vowels: AA AE AH AO AW AX AY EH ER EY IH IY IX OW OY UH UW UX             │
# │   (with stress digits 0/1/2 appended)                                          │
# │                                                                                │
# │ CMN_CMN (Pinyin-based):                                                        │
# │   Monophthongs: a e i o u v ii                                                 │
# │   Diphthongs/combos: ai ao ei ou ia iao ie iu iou ua uai ue uei uo va         │
# │   (all with tone digits 1-5)                                                   │
# │                                                                                │
# │ FRA_FRA (French IPA-ish):                                                      │
# │   Oral: a e i o u y AE E EU O OE                                               │
# │   Nasal: A~ E~ OE~ o~                                                          │
# │   Semi-vowels (treated as consonants): W J                                     │
# │                                                                                │
# │ GER_GER (German SAMPA-ish):                                                    │
# │   Short: a e i o u ae oe ue                                                    │
# │   Long: al el il ol ul ael oel uel                                             │
# │   Reduced: atu etu                                                             │
# │   Diphthongs: aI aU                                                            │
# │                                                                                │
# │ KOR_KOR (Korean):                                                              │
# │   Monophthongs: A AE E EO EU I O OE U UE                                      │
# │   On-glide diphthongs: iE iEO iO iU euI                                       │
# │   Off-glide diphthongs: oA uEO                                                │
# │                                                                                │
# │ RUS_RUS (Russian):                                                             │
# │   Monophthongs: a i i2 o u                                                     │
# │   Iotated (j+V, treated as diphthongs): jA jE jU                              │
# │                                                                                │
# │ SPA_SPA (Spanish):                                                             │
# │   Monophthongs: a e i o u                                                      │
# │   Stressed variants: a+ i+ o+ u+                                               │
# │   Diphthong: eU                                                                │
# │                                                                                │
# │ TUR_TUR (Turkish):                                                             │
# │   8 vowels: ab(=a) e i i2(=ı) o oe(=ö) u ue(=ü)                              │
# │                                                                                │
# │ VIE_VIE (Vietnamese):                                                          │
# │   Nuclei: a1 a2 a3 e1 e2 i o1 o2 o3 u1 u2                                    │
# │   Diphthongs: ai ao au ay ay3 eo eu ie2 ieu oa oi oi2 oi3 ua ua2 uu2 uy      │
# │   Plus combos: uoi3                                                            │
# │   (all suffixed with _T1.._T6 for tone)                                       │
# └─────────────────────────────────────────────────────────────────────────────────┘

import numpy as np
import soundfile as sf
import re as _re_vowel

# ── ARPAbet vowels (for all *_ENG tasks) ──
# These are the BASE labels (before stress digits).  is_vowel() strips
# digits before checking membership here.  After normalization, AH is
# split into AH0 (schwa /ə/) vs AH1/AH2 (strut /ʌ/) — see
# normalize_vowel_label() and ARPABET_TO_IPA for the full mapping.
ARPABET_VOWELS = {
    "AA", "AE", "AH", "AO", "AW", "AX", "AY",
    "EH", "ER", "EY",
    "IH", "IY", "IX",
    "OW", "OY",
    "UH", "UW", "UX",
}

# ── Mandarin (CMN_CMN) vowel nuclei — the base before tone digits ──
CMN_VOWEL_BASES = {
    "a", "e", "i", "o", "u", "v", "ii",
    "ai", "ao", "ei", "ou",
    "ia", "iao", "ie", "iu", "iou",
    "ua", "uai", "ue", "uei", "uo",
    "va",
}

# ── French (FRA_FRA) vowels ──
FRA_VOWELS = {
    "a", "e", "i", "o", "u", "y",
    "AE", "E", "EU", "O", "OE", "AX",
    "A~", "E~", "OE~", "o~",
}

# ── German (GER_GER) vowels ──
GER_VOWELS = {
    "a", "e", "i", "o", "u", "ae", "oe", "ue",
    "al", "el", "il", "ol", "ul", "ael", "oel", "uel",
    "atu", "etu",
    "aI", "aU",
}

# ── Korean (KOR_KOR) vowels ──
KOR_VOWELS = {
    "A", "AE", "E", "EO", "EU", "I", "O", "OE", "U", "UE",
    "iE", "iEO", "iO", "iU", "euI",
    "oA", "uEO",
}

# ── Russian (RUS_RUS) vowels ──
RUS_VOWELS = {"a", "i", "i2", "o", "u", "jA", "jE", "jU"}

# ── Spanish (SPA_SPA) vowels ──
SPA_VOWELS = {"a", "e", "i", "o", "u", "a+", "i+", "o+", "u+", "eU"}

# ── Turkish (TUR_TUR) vowels ──
TUR_VOWELS = {"ab", "e", "i", "i2", "o", "oe", "u", "ue"}

# ── Vietnamese (VIE_VIE) vowel nuclei (before _T<digit> tone suffix) ──
VIE_VOWEL_BASES = {
    "a1", "a2", "a3",
    "e1", "e2",
    "i",
    "o1", "o2", "o3",
    "u1", "u2",
    "ai", "ao", "au", "ay", "ay3",
    "eo", "eu",
    "ie2", "ieu",
    "oa", "oi", "oi2", "oi3",
    "ua", "ua2", "uu2", "uy",
    "uoi3",
}


def _detect_phone_system(task_language: str) -> str:
    """Return the phone system key based on the task language."""
    return "ARPABET" if task_language == "ENG" else task_language


def is_vowel(phone_label: str, task_language: str = "ENG") -> bool:
    """Classify a phone label as vowel or consonant.

    Handles all 10 phone systems in ALLSSTAR by dispatching on task_language.
    """
    system = _detect_phone_system(task_language)

    if system == "ARPABET":
        base = phone_label.rstrip("0123456789")
        return base in ARPABET_VOWELS

    if system == "CMN":
        base = phone_label.rstrip("12345")
        return base in CMN_VOWEL_BASES

    if system == "FRA":
        return phone_label in FRA_VOWELS

    if system == "GER":
        return phone_label in GER_VOWELS

    if system == "KOR":
        return phone_label in KOR_VOWELS

    if system == "RUS":
        return phone_label in RUS_VOWELS

    if system == "SPA":
        return phone_label in SPA_VOWELS

    if system == "TUR":
        return phone_label in TUR_VOWELS

    if system == "VIE":
        # Strip _T<digit> tone suffix, then check base
        base = _re_vowel.sub(r"_T\d$", "", phone_label)
        return base in VIE_VOWEL_BASES

    # Fallback: unknown system — try ARPAbet then common IPA
    base = phone_label.rstrip("0123456789")
    return base in ARPABET_VOWELS


def normalize_vowel_label(phone_label: str, task_language: str = "ENG") -> str:
    """Normalize a vowel label to its base form (strip stress/tone).

    Special case: AH retains its stress digit (AH0 = schwa /ə/,
    AH1/AH2 = strut /ʌ/) so they are not collapsed together.
    """
    system = _detect_phone_system(task_language)

    if system == "ARPABET":
        base = phone_label.rstrip("0123456789")
        if base == "AH" and len(phone_label) > len(base):
            return phone_label  # keep AH0, AH1, AH2 distinct
        return base
    if system == "CMN":
        return phone_label.rstrip("12345")
    if system == "VIE":
        return _re_vowel.sub(r"_T\d$", "", phone_label)
    return phone_label

# ── Load the WAV ──
wav_path = row["wav_path"]
task_lang = row["task_language"]
audio_data, sr = sf.read(wav_path)
if audio_data.ndim > 1:
    audio_data = audio_data[:, 0]  # mono
duration_s = len(audio_data) / sr

print(f"WAV: {wav_path}")
print(f"Task language: {task_lang}  →  phone system: {_detect_phone_system(task_lang)}")
print(f"Sample rate: {sr} Hz | Duration: {duration_s:.2f} s | Samples: {len(audio_data)}")
print()

# ── Slice each phoneme ──
phoneme_segments = []
for iv in phone_intervals:
    label = iv["text"]
    start_sample = int(iv["xmin"] * sr)
    end_sample = int(iv["xmax"] * sr)
    segment = audio_data[start_sample:end_sample]
    vow = is_vowel(label, task_lang)
    phoneme_segments.append({
        "label": label,
        "type": "vowel" if vow else "consonant",
        "xmin": iv["xmin"],
        "xmax": iv["xmax"],
        "duration_ms": (iv["xmax"] - iv["xmin"]) * 1000,
        "audio": segment,
        "sr": sr,
    })

n_vowels = sum(1 for s in phoneme_segments if s["type"] == "vowel")
n_consonants = len(phoneme_segments) - n_vowels

print(f"Total phoneme segments: {len(phoneme_segments)}")
print(f"  Vowels     : {n_vowels}")
print(f"  Consonants : {n_consonants}")
print()

# Show first 25 segments
print(f"{'#':>4}  {'Label':<12} {'Type':<11} {'Start':>8} {'End':>8} {'Dur(ms)':>8}  {'Samples':>8}")
print("-" * 72)
for j, seg in enumerate(phoneme_segments[:25]):
    print(f"{j:4d}  {seg['label']:<12} {seg['type']:<11} {seg['xmin']:8.3f} {seg['xmax']:8.3f} {seg['duration_ms']:8.1f}  {len(seg['audio']):8d}")
if len(phoneme_segments) > 25:
    print(f"  ... ({len(phoneme_segments) - 25} more segments)")

# Quick sanity check: vowel label distribution
from collections import Counter
vowel_counts = Counter(s["label"] for s in phoneme_segments if s["type"] == "vowel")
print(f"\nVowel label distribution ({len(vowel_counts)} unique):")
for lbl, cnt in vowel_counts.most_common():
    print(f"   {lbl:<12} {cnt:>4}")

In [ ]:
# ── Step 3: Extract F1/F2 for the SELECTED participant file, plot quadrilateral ──

%matplotlib widget

import parselmouth
from parselmouth.praat import call
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.patheffects as pe
matplotlib.rcParams.update({"figure.dpi": 100})

def extract_formants(audio_segment, sr, max_formant=5500, n_formants=5,
                     n_samples=5):
    """Extract F1 and F2 averaged over the middle third (33%–66%) of a segment.

    Samples *n_samples* equally-spaced time points in the middle third and
    returns the mean F1 and F2, ignoring any undefined (NaN) frames.
    This reduces coarticulation effects compared to a single midpoint measurement.
    """
    snd = parselmouth.Sound(audio_segment, sampling_frequency=sr)
    formant_obj = call(snd, "To Formant (burg)", 0.0, n_formants, max_formant, 0.025, 50.0)

    t_start = snd.duration / 3.0
    t_end   = snd.duration * 2.0 / 3.0
    times = np.linspace(t_start, t_end, n_samples)

    f1_vals, f2_vals = [], []
    for t in times:
        f1 = call(formant_obj, "Get value at time", 1, t, "Hertz", "Linear")
        f2 = call(formant_obj, "Get value at time", 2, t, "Hertz", "Linear")
        if not np.isnan(f1) and f1 > 0:
            f1_vals.append(f1)
        if not np.isnan(f2) and f2 > 0:
            f2_vals.append(f2)

    if not f1_vals or not f2_vals:
        return float("nan"), float("nan")
    return np.mean(f1_vals), np.mean(f2_vals)

def extract_vowel_formants_from_file(meta_row):
    """Extract all vowel formants from a single file. Returns list of dicts."""
    stem = meta_row["filename_stem"]
    tg_p = meta_row["textgrid_path"]
    wav_p = meta_row["wav_path"]
    if tg_p is None or wav_p is None:
        return []
    if not os.path.exists(wav_p):
        return []

    tg = parse_textgrid(tg_p)
    pt = None
    for t in tg:
        if "phone" in t["name"].lower():
            pt = t
            break
    if pt is None:
        pt = tg[-1]

    aud, sr_file = sf.read(wav_p)
    if aud.ndim > 1:
        aud = aud[:, 0]

    max_f = 5500 if meta_row["gender"] == "F" else 5000
    tl = meta_row["task_language"]
    results = []
    for iv in pt["intervals"]:
        lbl = iv["text"].strip()
        if not lbl or lbl in ("sil", "sp", "SIL", "SP"):
            continue
        if not is_vowel(lbl, tl):
            continue
        dur = iv["xmax"] - iv["xmin"]
        if dur < 0.03:
            continue

        s0 = int(iv["xmin"] * sr_file)
        s1 = int(iv["xmax"] * sr_file)
        seg = aud[s0:s1]
        try:
            f1, f2 = extract_formants(seg, sr_file, max_formant=max_f)
        except Exception:
            continue
        if np.isnan(f1) or np.isnan(f2) or f1 <= 0 or f2 <= 0:
            continue

        base_label = normalize_vowel_label(lbl, tl)
        results.append({
            "stem": stem,
            "participant_id": meta_row["participant_id"],
            "gender": meta_row["gender"],
            "vowel": base_label,
            "vowel_raw": lbl,
            "F1": f1,
            "F2": f2,
            "duration_ms": dur * 1000,
            "xmin": iv["xmin"],
            "xmax": iv["xmax"],
            "wav_path": wav_p,
            "textgrid_path": tg_p,
        })
    return results

# ── Process the single selected file ──
print(f"Selected file: {selected_stem}")
print(f"Participant: {row['participant_id']}  |  L1: {row['native_language']}  |  "
      f"Task lang: {row['task_language']}  |  Task: {row['task']}")
print("=" * 60)

single_formants = extract_vowel_formants_from_file(row)
df_single = pd.DataFrame(single_formants)
print(f"Vowel tokens with valid F1/F2: {len(df_single)}")
print(f"Unique vowel labels: {sorted(df_single['vowel'].unique())}")
print()
print(df_single.groupby("vowel")[["F1", "F2"]].agg(["mean", "std", "count"]).round(0))

# ── Full ARPAbet → IPA mapping ──
# Monophthongs (including AH split by stress and reduced-vowel variants)
# Diphthongs included for completeness but can be filtered downstream.
#
# ┌───────────┬───────┬─────────────────────────────────────────────────┐
# │ ARPAbet   │  IPA  │ Example word / notes                           │
# ├───────────┼───────┼─────────────────────────────────────────────────┤
# │ IY        │  i    │ fleece, beat                                    │
# │ IH        │  ɪ    │ kit, bit                                       │
# │ EH        │  ɛ    │ dress, bet                                     │
# │ AE        │  æ    │ trap, bat                                      │
# │ AA        │  ɑ    │ lot, father                                    │
# │ AO        │  ɔ    │ thought, caught                                │
# │ UH        │  ʊ    │ foot, put                                      │
# │ UW        │  u    │ goose, boot                                    │
# │ AH1/AH2   │  ʌ    │ strut, but (stressed)                          │
# │ AH0       │  ə    │ comma, about (unstressed schwa)                │
# │ ER        │  ɝ    │ nurse, bird (rhotacized)                       │
# │ AX        │  ə    │ schwa variant (same as AH0 in some systems)    │
# │ IX        │  ɨ    │ roses (reduced high-central; mapped to ɪ here) │
# │ UX        │  ʉ    │ reduced high-back (mapped to ʊ here)           │
# │ AW        │  aʊ   │ mouth, cow (diphthong)                         │
# │ AY        │  aɪ   │ price, buy (diphthong)                         │
# │ EY        │  eɪ   │ face, bay (diphthong)                          │
# │ OW        │  oʊ   │ goat, boat (diphthong)                         │
# │ OY        │  ɔɪ   │ choice, boy (diphthong)                        │
# └───────────┴───────┴─────────────────────────────────────────────────┘
ARPABET_TO_IPA = {
    # ── Monophthongs ──
    "IY":  "i",        # fleece
    "IH":  "\u026a",   # ɪ  kit
    "EH":  "\u025b",   # ɛ  dress
    "AE":  "\u00e6",   # æ  trap
    "AA":  "\u0251",   # ɑ  lot
    "AO":  "\u0254",   # ɔ  thought
    "UH":  "\u028a",   # ʊ  foot
    "UW":  "u",        # u  goose
    "AH1": "\u028c",   # ʌ  strut (primary stress)
    "AH2": "\u028c",   # ʌ  strut (secondary stress)
    "AH0": "\u0259",   # ə  schwa (unstressed)
    "ER":  "\u025d",   # ɝ  nurse (rhotacized mid-central)
    # ── Reduced variants (rare in ALLSSTAR) ──
    "AX":  "\u0259",   # ə  schwa variant
    "IX":  "\u026a",   # ɪ  reduced kit
    "UX":  "\u028a",   # ʊ  reduced foot
    # ── Diphthongs ──
    "AW":  "a\u028a",  # aʊ  mouth
    "AY":  "a\u026a",  # aɪ  price
    "EY":  "e\u026a",  # eɪ  face
    "OW":  "o\u028a",  # oʊ  goat
    "OY":  "\u0254\u026a",  # ɔɪ  choice
}

# Monophthongs to show on the vowel quadrilateral plot.
# Excludes diphthongs, schwa (ə), rhotacized (ɝ), and ɔ (thought).
IPA_MONOPHTHONGS = {"i", "\u026a", "\u025b", "\u00e6", "\u0251", "\u028a", "u", "\u028c"}

def add_ipa_column(df, task_lang):
    """Add an 'ipa_symbol' column mapped via ARPABET_TO_IPA, then keep only
    the monophthongs in IPA_MONOPHTHONGS for quadrilateral plotting.
    Returns filtered copy, or None if task_lang is not ENG / no matches."""
    if task_lang != "ENG":
        return None
    df_ipa = df.copy()
    df_ipa["ipa_symbol"] = df_ipa["vowel"].map(ARPABET_TO_IPA)
    df_ipa = df_ipa.dropna(subset=["ipa_symbol"])
    df_ipa = df_ipa[df_ipa["ipa_symbol"].isin(IPA_MONOPHTHONGS)]
    if df_ipa.empty:
        return None
    return df_ipa

def lobanov_normalize(df):
    """Lobanov (z-score) normalization per speaker.

    For each participant, F1 and F2 are converted to z-scores using that
    speaker's mean and SD computed over ALL their vowel tokens.
    Adds F1_norm and F2_norm columns; returns the dataframe (modified in place).
    """
    speaker_stats = df.groupby("participant_id")[["F1", "F2"]].agg(["mean", "std"])
    for formant in ["F1", "F2"]:
        col_mean = speaker_stats[(formant, "mean")]
        col_std  = speaker_stats[(formant, "std")]
        df[f"{formant}_norm"] = df.apply(
            lambda r: (r[formant] - col_mean[r["participant_id"]])
                      / col_std[r["participant_id"]]
            if col_std[r["participant_id"]] > 0 else 0.0,
            axis=1,
        )
    return df

def remove_outliers(df, threshold=2.5, f1_col="F1", f2_col="F2",
                    vowel_col="vowel"):
    """Remove per-vowel outliers based on Mahalanobis distance.

    Tokens whose Mahalanobis distance from their vowel-class centroid
    exceeds *threshold* are dropped.  Falls back to Euclidean (scaled by
    std) when a vowel class has fewer than 3 tokens.

    Returns (cleaned_df, n_removed).
    """
    from scipy.spatial.distance import mahalanobis as _mahal

    dists = np.full(len(df), np.nan)
    for v, grp in df.groupby(vowel_col):
        idx = grp.index
        X = grp[[f1_col, f2_col]].values
        mu = X.mean(axis=0)
        if len(X) >= 3:
            cov = np.cov(X, rowvar=False)
            try:
                cov_inv = np.linalg.inv(cov)
                for i, row_vals in zip(idx, X):
                    dists[df.index.get_loc(i)] = _mahal(row_vals, mu, cov_inv)
                continue
            except np.linalg.LinAlgError:
                pass
        # Fallback: z-score-like distance
        sd = X.std(axis=0)
        sd[sd == 0] = 1.0
        for i, row_vals in zip(idx, X):
            dists[df.index.get_loc(i)] = np.linalg.norm((row_vals - mu) / sd)

    mask = dists <= threshold
    n_removed = (~mask).sum()
    return df[mask].copy(), n_removed

# ── Normalize single-participant formants ──
OUTLIER_THRESHOLD = 2.5   # adjustable: Mahalanobis distance cutoff

df_single = lobanov_normalize(df_single)
df_single, n_out = remove_outliers(df_single, threshold=OUTLIER_THRESHOLD)
print(f"Outlier removal (threshold={OUTLIER_THRESHOLD}): {n_out} tokens removed, "
      f"{len(df_single)} remaining")

df_single_ipa = add_ipa_column(df_single, row["task_language"])
if df_single_ipa is not None:
    print(f"\n── IPA monophthong subset: {len(df_single_ipa)} tokens, "
          f"{sorted(df_single_ipa['ipa_symbol'].unique())} ──")
    print(df_single_ipa.groupby("ipa_symbol")[["F1", "F2"]].agg(["mean", "std", "count"]).round(0))

In [ ]:
# ── Vowel quadrilateral plot — SINGLE participant (interactive) ──

def _plot_vowel_row(df, vowel_col, axs, title_extra,
                    cmap_name="tab20", f1_col="F1", f2_col="F2"):
    """Plot one row of vowel space: scatter (left) + means with ellipses (right)."""
    ax_scatter, ax_ellipse = axs
    cmap = plt.cm.get_cmap(cmap_name)
    unique_vowels = sorted(df[vowel_col].unique())
    vowel_colors = {v: cmap(i / max(len(unique_vowels) - 1, 1))
                    for i, v in enumerate(unique_vowels)}

    is_norm = "norm" in f1_col
    unit = "Lobanov z" if is_norm else "Hz"
    f1_label = f"F1 ({unit})"
    f2_label = f"F2 ({unit})"

    for v in unique_vowels:
        sub = df[df[vowel_col] == v]
        ax_scatter.scatter(sub[f2_col], sub[f1_col], label=v, alpha=0.5, s=22,
                           color=vowel_colors[v], picker=True)
    ax_scatter.set_xlabel(f2_label)
    ax_scatter.set_ylabel(f1_label)
    ax_scatter.set_title(f"All vowel tokens{title_extra}")
    ax_scatter.invert_xaxis()
    ax_scatter.invert_yaxis()
    ax_scatter.legend(fontsize=7, ncol=2, loc="lower left")
    ax_scatter.grid(True, alpha=0.3)

    means  = df.groupby(vowel_col)[[f1_col, f2_col]].mean()
    stds   = df.groupby(vowel_col)[[f1_col, f2_col]].std()
    counts = df.groupby(vowel_col).size()

    for v in unique_vowels:
        if v not in means.index:
            continue
        c = vowel_colors[v]
        f1_m, f2_m = means.loc[v, f1_col], means.loc[v, f2_col]
        f1_s = stds.loc[v, f1_col] if not np.isnan(stds.loc[v, f1_col]) else 0
        f2_s = stds.loc[v, f2_col] if not np.isnan(stds.loc[v, f2_col]) else 0
        n = counts.loc[v]

        if f1_s > 0 and f2_s > 0:
            ell = matplotlib.patches.Ellipse(
                (f2_m, f1_m), width=2*f2_s, height=2*f1_s,
                fill=False, edgecolor=c, linewidth=1.2, alpha=0.6,
            )
            ax_ellipse.add_patch(ell)
        ax_ellipse.scatter(f2_m, f1_m, color=c, s=60, zorder=5,
                           edgecolors="k", linewidths=0.5)
        ax_ellipse.annotate(
            f"/{v}/ (n={n})", (f2_m, f1_m),
            textcoords="offset points", xytext=(6, -8), fontsize=8, fontweight="bold",
            color=c, path_effects=[pe.withStroke(linewidth=2, foreground="white")],
        )

    ax_ellipse.set_xlabel(f2_label)
    ax_ellipse.set_ylabel(f1_label)
    ax_ellipse.set_title(f"Vowel means ± 1 SD{title_extra}")
    ax_ellipse.invert_xaxis()
    ax_ellipse.invert_yaxis()
    ax_ellipse.grid(True, alpha=0.3)

    f2_range = df[f2_col].max() - df[f2_col].min()
    f1_range = df[f1_col].max() - df[f1_col].min()
    pad = 0.08
    ax_ellipse.set_xlim(df[f2_col].max() + pad * f2_range,
                        df[f2_col].min() - pad * f2_range)
    ax_ellipse.set_ylim(df[f1_col].max() + pad * f1_range,
                        df[f1_col].min() - pad * f1_range)


def plot_vowel_space(df, title_extra="", df_ipa=None, normalized=False):
    """Vowel space plot: 1×2 when no IPA data, 2×2 with an IPA-monophthong row otherwise."""
    f1c = "F1_norm" if normalized else "F1"
    f2c = "F2_norm" if normalized else "F2"
    nrows = 2 if df_ipa is not None else 1
    fig, axs = plt.subplots(nrows, 2, figsize=(10, 4 * nrows))
    if nrows == 1:
        axs = axs[np.newaxis, :]

    _plot_vowel_row(df, "vowel", axs[0],
                    title_extra + " (ARPAbet)" if df_ipa is not None else title_extra,
                    f1_col=f1c, f2_col=f2c)

    if df_ipa is not None:
        _plot_vowel_row(df_ipa, "ipa_symbol", axs[1],
                        title_extra + " (IPA monophthongs)", cmap_name="viridis",
                        f1_col=f1c, f2_col=f2c)

    return fig

# Plot for the single selected participant
pid = row["participant_id"]
fig = plot_vowel_space(df_single, title_extra=f" — Participant {pid}",
                       df_ipa=df_single_ipa)
fig.suptitle(
    f"Vowel Space — Participant {pid} ({row['gender']}) | "
    f"L1: {row['native_language']} | Task lang: {row['task_language']} | Task: {row['task']} | "
    f"{len(df_single)} tokens",
    fontsize=11, fontweight="bold",
)
fig.tight_layout()

In [ ]:
# ── Outlier inspector: find top-N outlier vowels and review spectrogram + audio ──

import ipywidgets as widgets
from IPython.display import display, Audio, clear_output
from scipy.spatial.distance import mahalanobis

# ---------------------------------------------------------------------------
# 1. Detect outliers via per-vowel Mahalanobis distance
# ---------------------------------------------------------------------------

def find_outliers(df, n=10, f1_col="F1", f2_col="F2", vowel_col="vowel"):
    """Rank all tokens by Mahalanobis distance from their vowel-class centroid.

    Falls back to Euclidean distance when a vowel class has fewer than 3 tokens
    (covariance matrix would be singular).
    """
    dists = np.full(len(df), np.nan)
    for v, grp in df.groupby(vowel_col):
        idx = grp.index
        X = grp[[f1_col, f2_col]].values
        mu = X.mean(axis=0)
        if len(X) >= 3:
            cov = np.cov(X, rowvar=False)
            try:
                cov_inv = np.linalg.inv(cov)
                for i, row_vals in zip(idx, X):
                    dists[df.index.get_loc(i)] = mahalanobis(row_vals, mu, cov_inv)
                continue
            except np.linalg.LinAlgError:
                pass
        for i, row_vals in zip(idx, X):
            dists[df.index.get_loc(i)] = np.linalg.norm(row_vals - mu)

    df = df.copy()
    df["mahal_dist"] = dists
    return df.sort_values("mahal_dist", ascending=False)

# ---------------------------------------------------------------------------
# 2. Audio loading helper (soundfile — consistent with the rest of the notebook)
# ---------------------------------------------------------------------------

_wav_cache = {}

def _load_wav_mono(wav_path):
    """Load a WAV file as mono float64 array + sample rate, with caching."""
    if wav_path not in _wav_cache:
        aud, sr = sf.read(wav_path, dtype="float64")
        if aud.ndim > 1:
            aud = aud[:, 0]
        _wav_cache[wav_path] = (aud, sr)
    return _wav_cache[wav_path]

# ---------------------------------------------------------------------------
# 3. Spectrogram + boundary drawing helpers (uses parselmouth)
# ---------------------------------------------------------------------------

def _draw_spectrogram_with_context(wav_path, xmin, xmax, ax,
                                   context_s=0.15, dynamic_range_db=50):
    """Draw a broadband spectrogram centred on [xmin, xmax] with context."""
    aud, sr = _load_wav_mono(wav_path)
    t0 = max(0.0, xmin - context_s)
    t1 = min(len(aud) / sr, xmax + context_s)
    seg = aud[int(t0 * sr):int(t1 * sr)]

    snd = parselmouth.Sound(seg, sampling_frequency=sr)
    spec = snd.to_spectrogram(window_length=0.005, maximum_frequency=5500)

    X, Y = spec.x_grid(), spec.y_grid()
    sg_db = 10 * np.log10(spec.values + 1e-20)
    floor = sg_db.max() - dynamic_range_db
    sg_db = np.clip(sg_db, floor, None)

    ax.pcolormesh(X + t0, Y, sg_db, cmap="afmhot", shading="auto")
    ax.axvline(xmin, color="cyan", ls="--", lw=1.5, label="MFA boundary")
    ax.axvline(xmax, color="cyan", ls="--", lw=1.5)
    ax.set_ylim(0, 5500)
    ax.set_ylabel("Frequency (Hz)")
    ax.set_xlabel("Time (s)")
    return t0, t1


def _draw_formant_track(wav_path, xmin, xmax, ax, gender,
                        context_s=0.15):
    """Overlay F1–F3 tracks on an existing spectrogram axis."""
    aud, sr = _load_wav_mono(wav_path)
    t0 = max(0.0, xmin - context_s)
    t1 = min(len(aud) / sr, xmax + context_s)
    seg = aud[int(t0 * sr):int(t1 * sr)]

    snd = parselmouth.Sound(seg, sampling_frequency=sr)
    max_f = 5500 if gender == "F" else 5000
    formant = call(snd, "To Formant (burg)", 0.0, 5, max_f, 0.025, 50.0)
    times = np.linspace(0, snd.duration, 200)
    colours = ["#00FFFF", "#00FF00", "#FFFF00"]
    for fn, c in zip([1, 2, 3], colours):
        vals = [call(formant, "Get value at time", fn, t, "Hertz", "Linear")
                for t in times]
        ax.plot(times + t0, vals, color=c, lw=1, alpha=0.8,
                label=f"F{fn}" if fn <= 3 else "")


def _get_neighboring_phones(tg_path, xmin, xmax, n_context=2):
    """Return a few phone labels surrounding the target vowel from the TextGrid."""
    tiers = parse_textgrid(tg_path)
    phone_tier = None
    for t in tiers:
        if "phone" in t["name"].lower():
            phone_tier = t
            break
    if phone_tier is None:
        phone_tier = tiers[-1]

    ivs = phone_tier["intervals"]
    target_idx = None
    for i, iv in enumerate(ivs):
        if abs(iv["xmin"] - xmin) < 0.001 and abs(iv["xmax"] - xmax) < 0.001:
            target_idx = i
            break
    if target_idx is None:
        return "(?)"

    lo = max(0, target_idx - n_context)
    hi = min(len(ivs), target_idx + n_context + 1)
    parts = []
    for j in range(lo, hi):
        lbl = ivs[j]["text"].strip() or "sil"
        if j == target_idx:
            lbl = f"[{lbl}]"
        parts.append(lbl)
    return " ".join(parts)

# ---------------------------------------------------------------------------
# 4. Vowel-space highlight subplot
# ---------------------------------------------------------------------------

def _draw_vowel_space_highlight(df, r, ax, f1_col="F1", f2_col="F2",
                                vowel_col="vowel"):
    """Draw the full vowel scatter with the current outlier highlighted."""
    cmap = plt.cm.get_cmap("tab20")
    unique_vowels = sorted(df[vowel_col].unique())
    vowel_colors = {v: cmap(i / max(len(unique_vowels) - 1, 1))
                    for i, v in enumerate(unique_vowels)}

    for v in unique_vowels:
        sub = df[df[vowel_col] == v]
        ax.scatter(sub[f2_col], sub[f1_col], alpha=0.25, s=16,
                   color=vowel_colors[v], label=v)

    ax.scatter(r[f2_col], r[f1_col], s=200, facecolors="none",
               edgecolors="red", linewidths=2.5, zorder=10)
    ax.scatter(r[f2_col], r[f1_col], s=40, color="red", zorder=11,
               label=f"outlier /{r[vowel_col]}/")

    ax.invert_xaxis()
    ax.invert_yaxis()
    is_norm = "norm" in f1_col
    unit = "Lobanov z" if is_norm else "Hz"
    ax.set_xlabel(f"F2 ({unit})")
    ax.set_ylabel(f"F1 ({unit})")
    ax.set_title("Vowel space (outlier = red ring)")
    ax.legend(fontsize=6, ncol=2, loc="lower left")
    ax.grid(True, alpha=0.3)

# ---------------------------------------------------------------------------
# 5. Interactive inspector widget
# ---------------------------------------------------------------------------

def launch_outlier_inspector(df, meta_row=None, n_outliers=10,
                             f1_col="F1", f2_col="F2", vowel_col="vowel"):
    """Launch an ipywidgets GUI to step through the top-N formant outliers.

    Shows three panels side by side:
      Left  — vowel quadrilateral with the outlier highlighted (red ring)
      Right — spectrogram with MFA boundaries + formant tracks
      Below — audio playback widget
    """
    ranked = find_outliers(df, n=n_outliers, f1_col=f1_col, f2_col=f2_col,
                           vowel_col=vowel_col)
    top = ranked.head(n_outliers).reset_index(drop=True)

    slider = widgets.IntSlider(
        value=0, min=0, max=len(top) - 1, step=1,
        description="Outlier #", continuous_update=False,
        layout=widgets.Layout(width="400px"),
    )
    flag_btn = widgets.ToggleButton(
        value=False, description="Flag as bad",
        button_style="danger", icon="flag",
    )
    out_plot = widgets.Output()
    out_audio = widgets.Output()
    out_info = widgets.Output()

    flagged_indices = set()

    def _render(idx):
        r = top.iloc[idx]

        with out_info:
            clear_output(wait=True)
            ctx = _get_neighboring_phones(r["textgrid_path"], r["xmin"], r["xmax"])
            print(f"── Outlier {idx + 1}/{len(top)}  (Mahal dist = {r['mahal_dist']:.2f}) ──")
            print(f"  Vowel : /{r[vowel_col]}/  (raw: {r['vowel_raw']})")
            print(f"  F1={r[f1_col]:.0f} Hz   F2={r[f2_col]:.0f} Hz   "
                  f"dur={r['duration_ms']:.0f} ms")
            print(f"  File  : {r['stem']}")
            print(f"  Time  : {r['xmin']:.3f} – {r['xmax']:.3f} s")
            print(f"  Phones: {ctx}")

        with out_plot:
            clear_output(wait=True)
            fig_insp, (ax_vowel, ax_spec) = plt.subplots(
                1, 2, figsize=(12, 3.5),
                gridspec_kw={"width_ratios": [1, 1.5]})

            _draw_vowel_space_highlight(df, r, ax_vowel,
                                        f1_col=f1_col, f2_col=f2_col,
                                        vowel_col=vowel_col)

            _draw_spectrogram_with_context(
                r["wav_path"], r["xmin"], r["xmax"], ax_spec)
            _draw_formant_track(
                r["wav_path"], r["xmin"], r["xmax"], ax_spec, r["gender"])
            ax_spec.set_title(
                f"/{r[vowel_col]}/ — {r['stem']}  "
                f"[{r['xmin']:.3f}–{r['xmax']:.3f} s]",
                fontsize=10)
            ax_spec.legend(loc="upper right", fontsize=7, framealpha=0.7)
            fig_insp.tight_layout()
            plt.show()

        with out_audio:
            clear_output(wait=True)
            context_s = 0.15
            aud, sr = _load_wav_mono(r["wav_path"])
            t0 = max(0, r["xmin"] - context_s)
            t1 = min(len(aud) / sr, r["xmax"] + context_s)
            s0, s1 = int(t0 * sr), int(t1 * sr)
            clip = aud[s0:s1]
            print(f"  Audio clip: {t0:.3f}–{t1:.3f} s  "
                  f"({len(clip)} samples @ {sr} Hz, "
                  f"peak={np.abs(clip).max():.4f})")
            display(Audio(data=clip, rate=sr))

        flag_btn.value = idx in flagged_indices

    def _on_slider(change):
        _render(change["new"])

    def _on_flag(change):
        if change["new"]:
            flagged_indices.add(slider.value)
        else:
            flagged_indices.discard(slider.value)

    slider.observe(_on_slider, names="value")
    flag_btn.observe(_on_flag, names="value")

    panel = widgets.VBox([
        widgets.HBox([slider, flag_btn]),
        out_info,
        out_plot,
        out_audio,
    ])
    display(panel)
    _render(0)

    return top, flagged_indices

# ---------------------------------------------------------------------------
# 6. Run on the single-participant data
# ---------------------------------------------------------------------------

# Sanity-check: make sure the new columns exist
assert "xmin" in df_single.columns, (
    "df_single is missing 'xmin' — please re-run the formant extraction cell (cell above) first!"
)
print(f"wav_path in df_single: {df_single['wav_path'].iloc[0]}")
print(f"Time range check — first token: {df_single['xmin'].iloc[0]:.3f}–{df_single['xmax'].iloc[0]:.3f} s\n")
print("Top-10 formant outliers (by Mahalanobis distance from vowel centroid):\n")
top_outliers, flagged = launch_outlier_inspector(
    df_single, meta_row=row, n_outliers=10)

In [ ]:
# ── Vowel quadrilateral — SINGLE participant, ALL tasks combined ──
# Pull together every task for the same participant + task language,
# so we can see their full vowel space regardless of which task produced the data.

pid = row["participant_id"]
tl  = row["task_language"]

all_tasks_mask = (
    (file_metadata["participant_id"] == pid) &
    (file_metadata["task_language"] == tl)
)
all_tasks_subset = file_metadata[all_tasks_mask].copy()
tasks_found = sorted(all_tasks_subset["task"].unique())
print(f"Participant {pid} — task language {tl}")
print(f"Tasks found: {tasks_found}  ({len(all_tasks_subset)} files)")
print("=" * 60)

all_tasks_formants = []
for _, mrow in all_tasks_subset.iterrows():
    fmts = extract_vowel_formants_from_file(mrow)
    for d in fmts:
        d["task"] = mrow["task"]
    all_tasks_formants.extend(fmts)
    print(f"  {mrow['filename_stem']}: {len(fmts)} vowel formants")

df_all_tasks = pd.DataFrame(all_tasks_formants)
df_all_tasks = lobanov_normalize(df_all_tasks)
df_all_tasks, n_out = remove_outliers(df_all_tasks, threshold=OUTLIER_THRESHOLD)
print(f"Outlier removal (threshold={OUTLIER_THRESHOLD}): {n_out} removed")

df_all_tasks_ipa = add_ipa_column(df_all_tasks, tl)

print(f"\nTotal vowel tokens across all tasks: {len(df_all_tasks)}")
print(f"Tasks: {sorted(df_all_tasks['task'].unique())}")
print(f"Unique vowel labels: {sorted(df_all_tasks['vowel'].unique())}")
print()
print(df_all_tasks.groupby("vowel")[["F1", "F2"]].agg(["mean", "std", "count"]).round(0))
if df_all_tasks_ipa is not None:
    print(f"\n── IPA monophthong subset: {len(df_all_tasks_ipa)} tokens ──")
    print(df_all_tasks_ipa.groupby("ipa_symbol")[["F1", "F2"]].agg(["mean", "std", "count"]).round(0))

# Plot combined across all tasks
task_list_str = "/".join(tasks_found)
fig = plot_vowel_space(
    df_all_tasks,
    title_extra=f" — Participant {pid} (all tasks: {task_list_str})",
    df_ipa=df_all_tasks_ipa,
)
fig.suptitle(
    f"Vowel Space — Participant {pid} ({row['gender']}) | "
    f"L1: {row['native_language']} | Task lang: {tl} | Tasks: {task_list_str} | "
    f"{len(df_all_tasks)} tokens",
    fontsize=11, fontweight="bold",
)
fig.tight_layout()
plt.show()

In [ ]:
# ── Step 4: Extract F1/F2 for ALL subjects in the same folder ──
# Same L1, same task language, same task type — but all participants

l1 = row["native_language"]
task_lang = row["task_language"]
task = row["task"]

folder_mask = (
    (file_metadata["native_language"] == l1) &
    (file_metadata["task_language"] == task_lang) &
    (file_metadata["task"] == task)
)
folder_subset = file_metadata[folder_mask].copy()
print(f"Folder: L1={l1}, Task lang={task_lang}, Task={task}")
print(f"Files to process: {len(folder_subset)}")
print("=" * 60)

all_folder_formants = []

for idx_row, meta_row in folder_subset.iterrows():
    stem = meta_row["filename_stem"]
    formants = extract_vowel_formants_from_file(meta_row)
    all_folder_formants.extend(formants)
    print(f"  {stem}: {len(formants)} vowel formants")

df_folder = pd.DataFrame(all_folder_formants)
df_folder = lobanov_normalize(df_folder)
df_folder, n_out = remove_outliers(df_folder, threshold=OUTLIER_THRESHOLD)
print(f"\nOutlier removal (threshold={OUTLIER_THRESHOLD}): {n_out} removed")
print(f"Total vowel tokens with valid F1/F2: {len(df_folder)}")
print(f"Unique participants: {df_folder['participant_id'].nunique()}")
print(f"Unique vowel labels: {sorted(df_folder['vowel'].unique())}")
print()
print(df_folder.groupby("vowel")[["F1", "F2"]].agg(["mean", "std", "count"]).round(0))

df_folder_ipa = add_ipa_column(df_folder, task_lang)
if df_folder_ipa is not None:
    print(f"\n── IPA monophthong subset: {len(df_folder_ipa)} tokens, "
          f"{sorted(df_folder_ipa['ipa_symbol'].unique())} ──")
    print(df_folder_ipa.groupby("ipa_symbol")[["F1", "F2"]].agg(["mean", "std", "count"]).round(0))

# ── Gender split ──
gender_dfs = {}
gender_ipa_dfs = {}
for g in sorted(df_folder["gender"].unique()):
    gdf = df_folder[df_folder["gender"] == g]
    n_spk = gdf["participant_id"].nunique()
    gender_dfs[g] = gdf
    gender_ipa_dfs[g] = add_ipa_column(gdf, task_lang)
    print(f"\n{'─'*60}")
    print(f"Gender: {g}  |  {n_spk} speakers  |  {len(gdf)} tokens")
    print(gdf.groupby("vowel")[["F1", "F2"]].agg(["mean", "std", "count"]).round(0))
    if gender_ipa_dfs[g] is not None:
        print(f"\n  IPA subset: {len(gender_ipa_dfs[g])} tokens")
        print(gender_ipa_dfs[g].groupby("ipa_symbol")[["F1", "F2"]].agg(["mean", "std", "count"]).round(0))

In [ ]:
# ── Vowel quadrilateral plot — ALL subjects in folder (interactive) ──

# Combined plot (all genders)
n_speakers = df_folder["participant_id"].nunique()
fig = plot_vowel_space(df_folder, title_extra=f" — All {n_speakers} speakers",
                       df_ipa=df_folder_ipa, normalized=True)
fig.suptitle(
    f"Vowel Space — L1: {l1} | Task lang: {task_lang} | Task: {task} | "
    f"{n_speakers} speakers | {len(df_folder)} tokens",
    fontsize=11, fontweight="bold",
)
fig.tight_layout()
plt.show()

In [ ]:
# ── Vowel quadrilateral plot — Female speakers only ──

if "F" in gender_dfs:
    gdf = gender_dfs["F"]
    n_spk = gdf["participant_id"].nunique()
    fig_f = plot_vowel_space(gdf, title_extra=f" — Female ({n_spk} speakers)",
                             df_ipa=gender_ipa_dfs["F"], normalized=True)
    fig_f.suptitle(
        f"Vowel Space (Female) — L1: {l1} | Task lang: {task_lang} | "
        f"Task: {task} | {n_spk} speakers | {len(gdf)} tokens",
        fontsize=11, fontweight="bold",
    )
    fig_f.tight_layout()
    plt.show()
else:
    print("No female speakers in this subset.")

In [ ]:
# ── Vowel quadrilateral plot — Male speakers only ──

if "M" in gender_dfs:
    gdf = gender_dfs["M"]
    n_spk = gdf["participant_id"].nunique()
    fig_m = plot_vowel_space(gdf, title_extra=f" — Male ({n_spk} speakers)",
                             df_ipa=gender_ipa_dfs["M"], normalized=True)
    fig_m.suptitle(
        f"Vowel Space (Male) — L1: {l1} | Task lang: {task_lang} | "
        f"Task: {task} | {n_spk} speakers | {len(gdf)} tokens",
        fontsize=11, fontweight="bold",
    )
    fig_m.tight_layout()
    plt.show()
else:
    print("No male speakers in this subset.")

In [ ]:
# ── Step 5: Extract F1/F2 for ALL subjects × ALL tasks (same L1 & task language) ──
# Combines every task folder that shares the selected L1 and task language,
# giving a single pooled dataset that spans DHR, HT1, HT2, LPP, NWS, etc.

l1        = row["native_language"]
task_lang = row["task_language"]

all_mask = (
    (file_metadata["native_language"] == l1) &
    (file_metadata["task_language"] == task_lang)
)
all_subset = file_metadata[all_mask].copy()
all_tasks = sorted(all_subset["task"].unique())
print(f"L1={l1}, Task lang={task_lang}")
print(f"Tasks to combine: {all_tasks}  ({len(all_subset)} files total)")
print("=" * 60)

all_formants = []
for _, mrow in all_subset.iterrows():
    fmts = extract_vowel_formants_from_file(mrow)
    for d in fmts:
        d["task"] = mrow["task"]
    all_formants.extend(fmts)
    print(f"  {mrow['filename_stem']}: {len(fmts)} vowel formants")

df_all = pd.DataFrame(all_formants)
df_all = lobanov_normalize(df_all)
df_all, n_out = remove_outliers(df_all, threshold=OUTLIER_THRESHOLD)
print(f"Outlier removal (threshold={OUTLIER_THRESHOLD}): {n_out} removed")

df_all_ipa = add_ipa_column(df_all, task_lang)

n_spk_all = df_all["participant_id"].nunique()
task_list_str = "/".join(all_tasks)

print(f"\nTotal tokens: {len(df_all)} | Speakers: {n_spk_all} | Tasks: {task_list_str}")
print(f"Unique vowel labels: {sorted(df_all['vowel'].unique())}")
print()
print(df_all.groupby("vowel")[["F1", "F2"]].agg(["mean", "std", "count"]).round(0))
if df_all_ipa is not None:
    print(f"\n── IPA monophthong subset: {len(df_all_ipa)} tokens ──")
    print(df_all_ipa.groupby("ipa_symbol")[["F1", "F2"]].agg(["mean", "std", "count"]).round(0))

# ── Gender split (all tasks) ──
gender_all_dfs = {}
gender_all_ipa_dfs = {}
for g in sorted(df_all["gender"].unique()):
    gdf = df_all[df_all["gender"] == g]
    n_spk = gdf["participant_id"].nunique()
    gender_all_dfs[g] = gdf
    gender_all_ipa_dfs[g] = add_ipa_column(gdf, task_lang)
    print(f"\n{'─'*60}")
    print(f"Gender: {g}  |  {n_spk} speakers  |  {len(gdf)} tokens")
    print(gdf.groupby("vowel")[["F1", "F2"]].agg(["mean", "std", "count"]).round(0))
    if gender_all_ipa_dfs[g] is not None:
        print(f"\n  IPA subset: {len(gender_all_ipa_dfs[g])} tokens")
        print(gender_all_ipa_dfs[g].groupby("ipa_symbol")[["F1", "F2"]].agg(["mean", "std", "count"]).round(0))

In [ ]:
# ── Vowel quadrilateral — ALL subjects, ALL tasks combined ──

fig = plot_vowel_space(
    df_all,
    title_extra=f" — All {n_spk_all} speakers, tasks: {task_list_str}",
    df_ipa=df_all_ipa,
    normalized=True,
)
fig.suptitle(
    f"Vowel Space (all tasks) — L1: {l1} | Task lang: {task_lang} | "
    f"Tasks: {task_list_str} | {n_spk_all} speakers | {len(df_all)} tokens",
    fontsize=11, fontweight="bold",
)
fig.tight_layout()
plt.show()

In [ ]:
# ── Vowel quadrilateral — Female speakers, ALL tasks combined ──

if "F" in gender_all_dfs:
    gdf = gender_all_dfs["F"]
    n_spk = gdf["participant_id"].nunique()
    fig_f = plot_vowel_space(
        gdf,
        title_extra=f" — Female ({n_spk} speakers), tasks: {task_list_str}",
        df_ipa=gender_all_ipa_dfs["F"],
        normalized=True,
    )
    fig_f.suptitle(
        f"Vowel Space (Female, all tasks) — L1: {l1} | Task lang: {task_lang} | "
        f"Tasks: {task_list_str} | {n_spk} speakers | {len(gdf)} tokens",
        fontsize=11, fontweight="bold",
    )
    fig_f.tight_layout()
    plt.show()
else:
    print("No female speakers in this subset.")

In [ ]:
# ── Vowel quadrilateral — Male speakers, ALL tasks combined ──

if "M" in gender_all_dfs:
    gdf = gender_all_dfs["M"]
    n_spk = gdf["participant_id"].nunique()
    fig_m = plot_vowel_space(
        gdf,
        title_extra=f" — Male ({n_spk} speakers), tasks: {task_list_str}",
        df_ipa=gender_all_ipa_dfs["M"],
        normalized=True,
    )
    fig_m.suptitle(
        f"Vowel Space (Male, all tasks) — L1: {l1} | Task lang: {task_lang} | "
        f"Tasks: {task_list_str} | {n_spk} speakers | {len(gdf)} tokens",
        fontsize=11, fontweight="bold",
    )
    fig_m.tight_layout()
    plt.show()
else:
    print("No male speakers in this subset.")